# Description-Guided VadCLIP Fine-Tuning On Colab

This notebook is a runner for the Python files in `VadCLIP/src`. It does not duplicate the model source code.

## 1. Mount Drive And Configure Paths

In [ ]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT
DESCRIPTION_JSON = PROJECT_ROOT / 'code' / 'ucf_gpt_video_descriptions.json'
USE_VADCLIP_CHECKPOINT = False
PRETRAINED_MODEL = PROJECT_ROOT / 'model_ucf.pth'

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)

print('Project root:', PROJECT_ROOT)
print('Source dir:', SRC_DIR)
print('Drive feature root:', DRIVE_FEATURE_ROOT)
print('Feature root:', FEATURE_ROOT)
print('Use VadCLIP checkpoint:', USE_VADCLIP_CHECKPOINT)

## 2. Install Dependencies

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib


## 3. Check Required Files

In [ ]:
required_paths = [
    SRC_DIR / 'model_description.py',
    SRC_DIR / 'ucf_train_description.py',
    SRC_DIR / 'ucf_train_class_semantic.py',
    SRC_DIR / 'ucf_option_class_semantic.py',
    SRC_DIR / 'ucf_train_class_prototype.py',
    SRC_DIR / 'ucf_option_class_prototype.py',
    SRC_DIR / 'ucf_evaluate.py',
    SRC_DIR / 'ucf_analyze_checkpoints.py',
    SRC_DIR / 'utils' / 'dataset_description.py',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv',
    DESCRIPTION_JSON,
    PROJECT_ROOT / 'code' / 'ucf_class_description_prototypes_vadclip_train.json',
]
if USE_VADCLIP_CHECKPOINT:
    required_paths.append(PRETRAINED_MODEL)

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
if not DRIVE_FEATURE_ROOT.exists() and available_feature_archive is None:
    archive_names = ', '.join(path.name for path in DRIVE_FEATURE_ARCHIVES)
    raise FileNotFoundError(f'Missing UCFClipFeatures folder or one archive: {archive_names}')

print('All required files exist.')

## 4. Copy Features To Local Runtime

In [ ]:
import shutil
import subprocess
import time

subprocess.run(['df', '-h', '/content'], check=False)
available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
if available_feature_archive is not None:
    subprocess.run(['du', '-sh', str(available_feature_archive)], check=False)
else:
    subprocess.run(['du', '-sh', str(DRIVE_FEATURE_ROOT)], check=False)

start = time.time()
if available_feature_archive is not None:
    local_archive = Path('/content') / available_feature_archive.name
    if not local_archive.exists() or local_archive.stat().st_size != available_feature_archive.stat().st_size:
        print('Copying archive to local runtime:', available_feature_archive)
        shutil.copy2(available_feature_archive, local_archive)
    else:
        print('Local archive already exists with matching size:', local_archive)

    print('Extracting archive:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)

    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive must contain top-level folder UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run([
            'rsync', '-ah', '--info=progress2',
            f'{DRIVE_FEATURE_ROOT}/', f'{LOCAL_FEATURE_ROOT}/'
        ], check=True)
    else:
        subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'Prepared local features in {(time.time() - start) / 60:.1f} minutes')
print('Feature root:', FEATURE_ROOT)


## 5. Check Feature Coverage

In [ ]:
import csv
from collections import Counter

def check_feature_coverage(csv_path, feature_root, preview=30):
    rows = list(csv.DictReader(open(csv_path, encoding='utf-8')))
    missing = []
    for row in rows:
        path = feature_root / row['path']
        if not path.exists():
            missing.append(row)

    print(f'{csv_path.name}: rows={len(rows)}, missing_files={len(missing)}')
    if missing:
        print('Missing by label:', dict(Counter(row['label'] for row in missing)))
        for row in missing[:preview]:
            print(f"  {row.get('video_id', '')},{row['label']},{row['path']}")
        if len(missing) > preview:
            print(f'  ... and {len(missing) - preview} more')
        raise FileNotFoundError(f'{csv_path.name} references missing feature files. Re-upload UCFClipFeatures or rebuild the list from available files.')

for csv_path in [
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv',
]:
    check_feature_coverage(csv_path, FEATURE_ROOT)


## 6. Smoke Test Dataset

In [ ]:
from utils.dataset_description import UCFDescriptionDataset

label_map = {
    'Normal': 'normal', 'Abuse': 'abuse', 'Arrest': 'arrest', 'Arson': 'arson',
    'Assault': 'assault', 'Burglary': 'burglary', 'Explosion': 'explosion',
    'Fighting': 'fighting', 'RoadAccidents': 'roadAccidents', 'Robbery': 'robbery',
    'Shooting': 'shooting', 'Shoplifting': 'shoplifting', 'Stealing': 'stealing',
    'Vandalism': 'vandalism'
}

train_list = PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv'
dataset = UCFDescriptionDataset(256, str(train_list), False, label_map, str(FEATURE_ROOT), str(DESCRIPTION_JSON), normal=False)
feature, label, length, video_id, description = dataset[0]
print('Dataset size:', len(dataset))
print('Feature shape:', feature.shape)
print('Label:', label)
print('Length:', length)
print('Video ID:', video_id)
print('Description:', description[:200])

## 7. Fine-Tune

In [ ]:
import shlex
import subprocess

train_description_cmd = [
    'python', 'ucf_train_description.py',
    '--feature-root', str(FEATURE_ROOT),
    '--description-json', str(DESCRIPTION_JSON),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-desc', '0.003',
    '--lambda-contrastive', '0.003',
    '--lambda-distill', '0.05',
    '--desc-loss-type', 'contrastive',
    '--desc-pooling', 'topk',
    '--top-k-ratio', '0.15',
    '--contrastive-temperature', '0.07',
    '--contrastive-samples', 'all',
    '--use-desc-projection', 'true',
    '--distill-baseline', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--trainable-scope', 'all',
    '--lr', '1e-5',
    '--max-epoch', '3',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--cache-description-embeddings', 'true',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_contrastive',
    '--output-model-path', 'model/model_ucf_description_contrastive.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_description_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_description_cmd))
subprocess.run(train_description_cmd, check=True)

## 8. Evaluate Baseline And Fine-Tuned Model


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_description_contrastive.pth" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 9. Analyze Checkpoints And Plot Diagnostics


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_contrastive" \
  --description-model-path "model/model_ucf_description_contrastive.pth" \
  --output-dir "{PROJECT_ROOT / 'code' / 'ucf_checkpoint_diagnostics_contrastive'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6


## 10. Class Semantic Fine-Tune


In [ ]:
train_class_semantic_cmd = [
    'python', 'ucf_train_class_semantic.py',
    '--feature-root', str(FEATURE_ROOT),
    '--description-json', str(DESCRIPTION_JSON),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-sem', '0.05',
    '--semantic-alpha', '0.2',
    '--semantic-temperature', '0.07',
    '--video-logit-pooling', 'topk',
    '--top-k-ratio', '0.15',
    '--lr', '1e-5',
    '--max-epoch', '3',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--cache-semantic-targets', 'true',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_class_semantic',
    '--output-model-path', 'model/model_ucf_class_semantic.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_class_semantic_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_class_semantic_cmd))
subprocess.run(train_class_semantic_cmd, check=True)


## 11. Evaluate Class Semantic Model


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_class_semantic.pth" \
  --description-model-type baseline \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 12. Analyze Class Semantic Checkpoints


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_class_semantic" \
  --description-model-path "model/model_ucf_class_semantic.pth" \
  --finetuned-model-type baseline \
  --output-dir "{PROJECT_ROOT / 'code' / 'ucf_checkpoint_diagnostics_class_semantic'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6


## 13. Class Prototype Fine-Tune From Scratch

This is the current recommended prototype run. It does not load `model_ucf.pth` for training, and it uses prototype MIL loss as a real semantic supervision branch.

In [ ]:
import shlex
import subprocess

train_class_prototype_cmd = [
    'python', 'ucf_train_class_prototype.py',
    '--feature-root', str(FEATURE_ROOT),
    '--prototype-json', str(PROJECT_ROOT / 'code' / 'ucf_class_description_prototypes_vadclip_train.json'),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-proto-mil', '1.0',
    '--lambda-proto', '0.05',
    '--prototype-temperature', '0.07',
    '--prototype-mode', 'centroid',
    '--lr', '2e-5',
    '--max-epoch', '10',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_class_prototype_scratch',
    '--output-model-path', 'model/model_ucf_class_prototype_scratch.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_class_prototype_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_class_prototype_cmd))
subprocess.run(train_class_prototype_cmd, check=True)


## 14. Evaluate Class Prototype Model


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_class_prototype_scratch.pth" \
  --description-model-type baseline \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 15. Analyze Class Prototype Checkpoints


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_class_prototype_scratch" \
  --description-model-path "model/model_ucf_class_prototype_scratch.pth" \
  --finetuned-model-type baseline \
  --output-dir "{PROJECT_ROOT / 'code' / 'ucf_checkpoint_diagnostics_class_prototype_scratch'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6
